# EE 451: Communications Systems
## Lesson 19 — Spread Spectrum & CDMA

### Learning Objectives
By the end of this lesson, you will be able to:
- Analyze DSSS processing gain and interference rejection
- Generate and evaluate PN sequences (m-sequences)
- Explain Walsh codes and orthogonality in CDMA
- Design multi-user CDMA systems
- Compare CDMA to TDMA and FDMA multiple access schemes

### Textbook Reference
Supplemental materials

In [ ]:
# Setup: Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 2

print("Setup complete! NumPy version:", np.__version__)

## Part 1: DSSS Processing Gain

**Direct-Sequence Spread Spectrum (DSSS)** multiplies the data signal by a
high-rate pseudo-noise (PN) code, spreading the signal bandwidth.

- **Chip rate** $R_c$: Rate of PN code
- **Bit rate** $R_b$: Rate of data
- **Processing gain**: $G_p = R_c / R_b$
- **Key benefit**: After despreading, the SNR improves by $G_p$

**GPS example:** $R_b = 50$ bps, $R_c = 1.023$ Mcps $\Rightarrow G_p = 20{,}460 \approx 43$ dB

In [ ]:
# === Part 1: DSSS Spreading and Despreading ===

# Parameters
Rb = 1000     # Data bit rate (bps)
chips_per_bit = 15   # Processing gain = 15 (~12 dB)
Rc = Rb * chips_per_bit  # Chip rate
Gp = chips_per_bit
Gp_dB = 10 * np.log10(Gp)

# Generate data bits
data_bits = np.array([1, 0, 1, 1, 0])
data_bipolar = 2 * data_bits - 1  # Map 0/1 to -1/+1

# Generate PN code (one period, length = chips_per_bit)
np.random.seed(42)
pn_code = 2 * np.random.randint(0, 2, chips_per_bit) - 1  # Random -1/+1

# Spread: each data bit multiplied by PN code
spread_signal = np.array([])
for bit in data_bipolar:
    spread_signal = np.concatenate([spread_signal, bit * pn_code])

# Spectrum comparison: data vs spread
# Upsample data to chip rate for fair comparison
data_upsampled = np.repeat(data_bipolar, chips_per_bit)

N = len(spread_signal)
freqs = fftfreq(N, 1/Rc)[:N//2]
data_spectrum = np.abs(fft(data_upsampled))[:N//2]
spread_spectrum = np.abs(fft(spread_signal))[:N//2]

# Normalize for comparison
data_spectrum_dB = 20 * np.log10(data_spectrum / np.max(data_spectrum) + 1e-10)
spread_spectrum_dB = 20 * np.log10(spread_spectrum / np.max(spread_spectrum) + 1e-10)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Time domain - data
t_chip = np.arange(N) / Rc * 1000  # ms
axes[0, 0].step(t_chip, data_upsampled, where='mid', color='C0', linewidth=2)
for i in range(1, len(data_bits)):
    axes[0, 0].axvline(i / Rb * 1000, color='gray', linestyle='--', alpha=0.5)
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].set_title('Original Data Signal', fontsize=13, fontweight='bold')
axes[0, 0].set_ylim(-1.5, 1.5)

# Time domain - spread
axes[0, 1].step(t_chip, spread_signal, where='mid', color='C1', linewidth=1)
for i in range(1, len(data_bits)):
    axes[0, 1].axvline(i / Rb * 1000, color='gray', linestyle='--', alpha=0.5)
axes[0, 1].set_ylabel('Amplitude')
axes[0, 1].set_title(f'DSSS Spread Signal (G_p = {Gp})', fontsize=13, fontweight='bold')
axes[0, 1].set_ylim(-1.5, 1.5)

# Spectrum - data
axes[1, 0].plot(freqs / 1000, data_spectrum_dB, color='C0')
axes[1, 0].set_xlabel('Frequency (kHz)')
axes[1, 0].set_ylabel('Magnitude (dB)')
axes[1, 0].set_title('Data Spectrum (narrow)', fontsize=13, fontweight='bold')
axes[1, 0].set_xlim(0, Rc / 1000)
axes[1, 0].set_ylim(-40, 5)
axes[1, 0].axvspan(0, Rb / 1000 * 2, alpha=0.15, color='C0', label=f'Data BW \u2248 {2*Rb/1000:.0f} kHz')
axes[1, 0].legend(fontsize=9)

# Spectrum - spread
axes[1, 1].plot(freqs / 1000, spread_spectrum_dB, color='C1')
axes[1, 1].set_xlabel('Frequency (kHz)')
axes[1, 1].set_ylabel('Magnitude (dB)')
axes[1, 1].set_title('Spread Spectrum (wide)', fontsize=13, fontweight='bold')
axes[1, 1].set_xlim(0, Rc / 1000)
axes[1, 1].set_ylim(-40, 5)
axes[1, 1].axvspan(0, Rc / 1000, alpha=0.15, color='C1', label=f'Spread BW \u2248 {2*Rc/1000:.0f} kHz')
axes[1, 1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"DSSS Parameters:")
print(f"  Data rate:      Rb = {Rb} bps")
print(f"  Chip rate:      Rc = {Rc} cps")
print(f"  Processing gain: Gp = {Gp} = {Gp_dB:.1f} dB")
print(f"  Data BW:        ~{2*Rb/1000:.0f} kHz")
print(f"  Spread BW:      ~{2*Rc/1000:.0f} kHz")
print(f"\nGPS C/A code: Gp = 1,023,000 / 50 = 20,460 = {10*np.log10(20460):.1f} dB")

## Part 2: PN Sequences and Autocorrelation

**Pseudo-Noise (PN) sequences** are deterministic but appear random.

**Maximum-length sequences (m-sequences):**
- Generated by a Linear Feedback Shift Register (LFSR)
- Length: $L = 2^n - 1$ (where $n$ = register length)
- Autocorrelation: $R(0) = 1$, $R(\tau \neq 0) = -1/L$
- Near-ideal for spread spectrum: sharp peak enables synchronization

In [ ]:
# === Part 2: m-Sequence Generation and Autocorrelation ===

def generate_m_sequence(n_stages, taps, initial_state=None):
    """Generate m-sequence from LFSR.
    n_stages: number of shift register stages
    taps: list of feedback tap positions (0-indexed)
    Returns bipolar sequence (-1/+1).
    """
    L = 2**n_stages - 1
    if initial_state is None:
        state = [1] * n_stages
    else:
        state = list(initial_state)
    seq = []
    for _ in range(L):
        seq.append(state[-1])
        feedback = 0
        for tap in taps:
            feedback ^= state[tap]
        state = [feedback] + state[:-1]
    return 2 * np.array(seq) - 1  # Convert 0/1 to -1/+1

# Generate m-sequences of different lengths
# n=3: L=7, taps [0,2] (x^3 + x + 1)
# n=5: L=31, taps [0,2] (x^5 + x^2 + 1)
# n=7: L=127, taps [0,3] (x^7 + x^3 + 1)
m_seq_7 = generate_m_sequence(3, [0, 2])
m_seq_31 = generate_m_sequence(5, [0, 2])
m_seq_127 = generate_m_sequence(7, [0, 3])

fig, axes = plt.subplots(3, 2, figsize=(14, 9))

for idx, (seq, label, L) in enumerate([
    (m_seq_7, 'n=3, L=7', 7),
    (m_seq_31, 'n=5, L=31', 31),
    (m_seq_127, 'n=7, L=127', 127)
]):
    # Sequence
    axes[idx, 0].step(range(len(seq)), seq, where='mid', linewidth=1.5 if L < 50 else 0.8)
    axes[idx, 0].set_ylabel('Value')
    axes[idx, 0].set_title(f'm-Sequence ({label})', fontsize=12, fontweight='bold')
    axes[idx, 0].set_ylim(-1.5, 1.5)
    if idx == 2:
        axes[idx, 0].set_xlabel('Chip index')
    
    # Autocorrelation
    autocorr = np.correlate(seq, seq, mode='full') / L
    lags = np.arange(-L + 1, L)
    axes[idx, 1].plot(lags, autocorr, linewidth=1.5 if L < 50 else 0.8)
    axes[idx, 1].axhline(-1/L, color='red', linestyle='--', alpha=0.6, label=f'-1/L = {-1/L:.3f}')
    axes[idx, 1].axhline(0, color='gray', linewidth=0.5)
    axes[idx, 1].set_ylabel('R(\u03c4)')
    axes[idx, 1].set_title(f'Autocorrelation ({label})', fontsize=12, fontweight='bold')
    axes[idx, 1].legend(fontsize=9, loc='upper right')
    if idx == 2:
        axes[idx, 1].set_xlabel('Lag \u03c4')

plt.tight_layout()
plt.show()

print("m-Sequence Properties:")
print(f"{'Stages (n)':<12} {'Length (L)':<12} {'R(0)':<8} {'R(\u03c4\u22600)':<12} {'Balance (+1/-1)'}")
print("-" * 60)
for n, seq, L in [(3, m_seq_7, 7), (5, m_seq_31, 31), (7, m_seq_127, 127)]:
    n_pos = np.sum(seq == 1)
    n_neg = np.sum(seq == -1)
    print(f"{n:<12} {L:<12} {1.0:<8.3f} {-1/L:<12.4f} {n_pos}/{n_neg}")

## Part 3: Walsh Codes and Orthogonality

**Walsh codes** (from the Hadamard matrix) provide perfect orthogonality
for synchronous CDMA systems.

**Hadamard construction:** Start with $H_1 = [1]$, then recursively:
$$H_{2N} = \begin{bmatrix} H_N & H_N \\ H_N & -H_N \end{bmatrix}$$

**Orthogonality:** $W_i \cdot W_j = 0$ for $i \neq j$, meaning each user's
signal is invisible to other users' receivers.

In [ ]:
# === Part 3: Walsh Code Generation and Orthogonality ===

def hadamard(n):
    """Generate n x n Hadamard matrix (n must be power of 2)."""
    if n == 1:
        return np.array([[1]])
    H_half = hadamard(n // 2)
    return np.block([[H_half, H_half],
                     [H_half, -H_half]])

# Generate Walsh-8 codes
N_walsh = 8
W = hadamard(N_walsh)

# Verify orthogonality: compute correlation matrix
corr_matrix = W @ W.T / N_walsh

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Walsh codes as waveforms
for i in range(N_walsh):
    axes[0].step(np.arange(N_walsh), W[i] + i * 2.5, where='mid', linewidth=1.5)
    axes[0].text(-0.8, i * 2.5, f'W{i}', fontsize=10, fontweight='bold', va='center')
axes[0].set_xlabel('Chip index')
axes[0].set_title(f'Walsh-{N_walsh} Codes', fontsize=13, fontweight='bold')
axes[0].set_yticks([])
axes[0].set_xlim(-1, N_walsh)

# Hadamard matrix visualization
im = axes[1].imshow(W, cmap='RdBu', vmin=-1, vmax=1, aspect='equal')
axes[1].set_xlabel('Chip index')
axes[1].set_ylabel('Code index')
axes[1].set_title(f'Hadamard Matrix H_{N_walsh}', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=axes[1], shrink=0.8)

# Correlation matrix
im2 = axes[2].imshow(corr_matrix, cmap='RdBu', vmin=-1, vmax=1, aspect='equal')
axes[2].set_xlabel('Code j')
axes[2].set_ylabel('Code i')
axes[2].set_title('Cross-Correlation: W_i \u00b7 W_j / N', fontsize=13, fontweight='bold')
plt.colorbar(im2, ax=axes[2], shrink=0.8)
# Annotate diagonal
for i in range(N_walsh):
    for j in range(N_walsh):
        val = corr_matrix[i, j]
        if abs(val) > 0.01:
            axes[2].text(j, i, f'{val:.0f}', ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.show()

print(f"Walsh-{N_walsh} Orthogonality Verification:")
print(f"Correlation matrix is identity: {np.allclose(corr_matrix, np.eye(N_walsh))}")
print(f"\nMax off-diagonal correlation: {np.max(np.abs(corr_matrix - np.diag(np.diag(corr_matrix)))):.6f}")
print(f"\nWalsh-4 codes (used in example):")
W4 = hadamard(4)
for i in range(4):
    signs = [''.join(['+1' if x == 1 else '-1' for x in W4[i]])]
    print(f"  W{i} = {W4[i].astype(int)}")

## Part 4: Multi-User CDMA Simulation

In CDMA, multiple users transmit simultaneously on the same frequency.
Each user is assigned a unique Walsh code. The receiver extracts a specific
user's data by correlating with that user's code.

**Transmitter (User $k$):** $s_k(t) = d_k \cdot W_k$

**Channel:** $r(t) = \sum_k s_k(t) + n(t)$

**Receiver (User $k$):** $\hat{d}_k = \text{sign}\left(\frac{1}{N} r \cdot W_k\right)$

**Near-far problem:** If one user is much closer (stronger), it can overwhelm
weaker users. Power control is essential.

In [ ]:
# === Part 4: Multi-User CDMA with 4 Users ===

N_users = 4
N_code = 4  # Walsh-4 code length
W4 = hadamard(N_code)

# Each user transmits a sequence of bits
N_data_bits = 8
np.random.seed(12)
user_data = np.random.randint(0, 2, (N_users, N_data_bits))
user_data_bipolar = 2 * user_data - 1

# Power levels (equal power = ideal)
user_power = np.array([1.0, 1.0, 1.0, 1.0])

# Spread each user's data and combine
total_chips = N_data_bits * N_code
composite = np.zeros(total_chips)
user_signals = np.zeros((N_users, total_chips))

for u in range(N_users):
    for b in range(N_data_bits):
        idx_s = b * N_code
        idx_e = (b + 1) * N_code
        user_signals[u, idx_s:idx_e] = user_power[u] * user_data_bipolar[u, b] * W4[u]
    composite += user_signals[u]

# Add noise
SNR_dB = 10
sig_power = np.mean(composite**2)
noise_power = sig_power / (10**(SNR_dB / 10))
noise = np.sqrt(noise_power) * np.random.randn(total_chips)
received = composite + noise

# Despread and detect for each user
detected = np.zeros((N_users, N_data_bits), dtype=int)
corr_values = np.zeros((N_users, N_data_bits))

for u in range(N_users):
    for b in range(N_data_bits):
        idx_s = b * N_code
        idx_e = (b + 1) * N_code
        corr = np.dot(received[idx_s:idx_e], W4[u]) / N_code
        corr_values[u, b] = corr
        detected[u, b] = 1 if corr > 0 else 0

# Plot
fig, axes = plt.subplots(N_users + 1, 1, figsize=(14, 10),
                         gridspec_kw={'height_ratios': [2] + [1]*N_users})

# Composite signal
chip_idx = np.arange(total_chips)
axes[0].step(chip_idx, received, where='mid', linewidth=0.8, color='C4')
for b in range(1, N_data_bits):
    axes[0].axvline(b * N_code, color='gray', linestyle='--', alpha=0.3)
axes[0].set_ylabel('Amplitude')
axes[0].set_title(f'CDMA: {N_users} Users, Walsh-{N_code} Codes (SNR = {SNR_dB} dB)',
                  fontsize=14, fontweight='bold')

# Each user's detected data
colors = ['C0', 'C1', 'C2', 'C3']
for u in range(N_users):
    for b in range(N_data_bits):
        color = colors[u]
        alpha = 0.5 if detected[u, b] == user_data[u, b] else 0.2
        axes[u+1].fill_between([b*N_code, (b+1)*N_code], detected[u, b],
                               alpha=alpha, color=color)
        match = '\u2713' if detected[u, b] == user_data[u, b] else '\u2717'
        axes[u+1].text((b+0.5)*N_code, 0.5, f'{detected[u,b]}',
                       ha='center', va='center', fontsize=9)
    errors_u = np.sum(detected[u] != user_data[u])
    axes[u+1].set_ylabel(f'User {u}', fontsize=11, fontweight='bold', color=colors[u])
    axes[u+1].set_ylim(-0.1, 1.1)
    axes[u+1].text(total_chips + 0.5, 0.5, f'{errors_u} err', fontsize=10, va='center')

axes[-1].set_xlabel('Chip index')
plt.tight_layout()
plt.show()

total_errors = np.sum(detected != user_data)
total_bits = N_users * N_data_bits
print(f"Multi-User CDMA Results ({N_users} users, {N_data_bits} bits each):")
for u in range(N_users):
    err = np.sum(detected[u] != user_data[u])
    print(f"  User {u}: TX={user_data[u]}  RX={detected[u]}  Errors={err}/{N_data_bits}")
print(f"\nTotal BER: {total_errors}/{total_bits} = {total_errors/total_bits:.4f}")

## Part 5: Near-Far Problem

The **near-far problem** occurs when users have unequal received power.
A strong user's signal leaks into a weak user's correlator because
real-world imperfections break perfect orthogonality.

**With equal power:** Walsh code orthogonality provides perfect separation.

**With unequal power:** Strong users corrupt weak users' detection.

**Solution:** Power control — the base station commands each user to
adjust transmit power so all signals arrive at roughly equal strength.

In [ ]:
# === Part 5: Near-Far Problem Demonstration ===

# Simulate with unequal power and timing offsets
# User 0 (weak), User 1 (strong) - power imbalance
power_ratios_dB = np.arange(0, 25, 1)  # Power ratio of strong/weak user
N_trials = 500
N_bits_trial = 20

ber_weak_user = np.zeros(len(power_ratios_dB))
ber_strong_user = np.zeros(len(power_ratios_dB))

for pi, pr_dB in enumerate(power_ratios_dB):
    power_ratio = 10**(pr_dB / 10)
    weak_errors = 0
    strong_errors = 0
    total = 0
    
    for trial in range(N_trials):
        # Two users with Walsh-4 codes
        d_weak = 2 * np.random.randint(0, 2, N_bits_trial) - 1
        d_strong = 2 * np.random.randint(0, 2, N_bits_trial) - 1
        
        for b in range(N_bits_trial):
            # Signals
            s_weak = 1.0 * d_weak[b] * W4[0]
            s_strong = np.sqrt(power_ratio) * d_strong[b] * W4[1]
            
            # Add small timing offset (1 chip shift for strong user)
            s_strong_shifted = np.roll(s_strong, 1) if pr_dB > 0 else s_strong
            
            # Composite + noise
            r = s_weak + s_strong_shifted
            r += 0.3 * np.random.randn(N_code)
            
            # Detect weak user
            corr_weak = np.dot(r, W4[0]) / N_code
            det_weak = 1 if corr_weak > 0 else -1
            if det_weak != d_weak[b]:
                weak_errors += 1
            
            # Detect strong user
            corr_strong = np.dot(r, W4[1]) / N_code
            det_strong = 1 if corr_strong > 0 else -1
            if det_strong != d_strong[b]:
                strong_errors += 1
            
            total += 1
    
    ber_weak_user[pi] = weak_errors / total
    ber_strong_user[pi] = strong_errors / total

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(power_ratios_dB, ber_weak_user + 1e-6, 'C1-o', markersize=4, label='Weak user (victim)')
ax.semilogy(power_ratios_dB, ber_strong_user + 1e-6, 'C0-s', markersize=4, label='Strong user')
ax.set_xlabel('Power Ratio: Strong/Weak (dB)', fontsize=13)
ax.set_ylabel('Bit Error Rate', fontsize=13)
ax.set_title('Near-Far Problem: BER vs Power Imbalance', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.set_ylim(1e-4, 0.6)
ax.axvline(10, color='red', linestyle='--', alpha=0.5)
ax.text(10.5, 0.2, '10 dB imbalance', fontsize=10, color='red')

plt.tight_layout()
plt.show()

print("Near-Far Problem Summary:")
print(f"{'Power Ratio (dB)':<20} {'Weak User BER':<18} {'Strong User BER'}")
print("-" * 55)
for dB_val in [0, 5, 10, 15, 20]:
    idx = np.argmin(np.abs(power_ratios_dB - dB_val))
    print(f"{dB_val:<20} {ber_weak_user[idx]:<18.4f} {ber_strong_user[idx]:.4f}")
print(f"\nSolution: Power control ensures equal received power at base station")

## Part 6: CDMA vs TDMA vs FDMA

Three approaches to sharing a communication channel among multiple users:

| Scheme | Separation | Advantage | Disadvantage |
|--------|-----------|-----------|-------------|
| **FDMA** | Frequency | Simple | Fixed allocation, guard bands |
| **TDMA** | Time | Flexible | Synchronization required |
| **CDMA** | Code | Soft capacity, no coordination | Near-far problem, self-interference |

In [ ]:
# === Part 6: Visual Comparison of Multiple Access Schemes ===

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['C0', 'C1', 'C2', 'C3']
user_labels = ['User 1', 'User 2', 'User 3', 'User 4']

# FDMA
for i in range(4):
    axes[0].fill_between([0, 8], i + 0.1, i + 0.9, color=colors[i], alpha=0.6)
    axes[0].text(4, i + 0.5, user_labels[i], ha='center', va='center',
                fontsize=11, fontweight='bold', color='white')
axes[0].set_xlabel('Time (slots)', fontsize=12)
axes[0].set_ylabel('Frequency (channels)', fontsize=12)
axes[0].set_title('FDMA', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, 8)
axes[0].set_ylim(0, 4)
axes[0].set_yticks([0.5, 1.5, 2.5, 3.5])
axes[0].set_yticklabels(['f\u2081', 'f\u2082', 'f\u2083', 'f\u2084'])

# TDMA
for slot in range(8):
    user_idx = slot % 4
    axes[1].fill_between([slot + 0.05, slot + 0.95], 0, 4,
                         color=colors[user_idx], alpha=0.6)
    axes[1].text(slot + 0.5, 2, user_labels[user_idx][:2], ha='center', va='center',
                fontsize=9, fontweight='bold', color='white', rotation=90)
axes[1].set_xlabel('Time (slots)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('TDMA', fontsize=14, fontweight='bold')
axes[1].set_xlim(0, 8)
axes[1].set_ylim(0, 4)
axes[1].set_yticks([2])
axes[1].set_yticklabels(['Full BW'])

# CDMA
for i in range(4):
    # All users use full time and frequency, separated by code
    axes[2].fill_between([0, 8], 0 + i*0.02, 4 - i*0.02,
                         color=colors[i], alpha=0.2)
axes[2].text(4, 2, 'All users\nAll time\nAll frequency\n\nSeparated by CODE',
            ha='center', va='center', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
# Legend
for i in range(4):
    axes[2].fill_between([], [], color=colors[i], alpha=0.5, label=user_labels[i])
axes[2].legend(loc='upper right', fontsize=9)
axes[2].set_xlabel('Time (slots)', fontsize=12)
axes[2].set_ylabel('Frequency', fontsize=12)
axes[2].set_title('CDMA', fontsize=14, fontweight='bold')
axes[2].set_xlim(0, 8)
axes[2].set_ylim(0, 4)
axes[2].set_yticks([2])
axes[2].set_yticklabels(['Full BW'])

plt.tight_layout()
plt.show()

print("Multiple Access Comparison:")
print(f"{'Scheme':<10} {'Separation':<15} {'Synchronization':<18} {'Capacity':<20} {'Example'}")
print("-" * 80)
print(f"{'FDMA':<10} {'Frequency':<15} {'Low':<18} {'Fixed (N channels)':<20} {'1G AMPS'}")
print(f"{'TDMA':<10} {'Time':<15} {'Frame-level':<18} {'Fixed (N slots)':<20} {'2G GSM'}")
print(f"{'CDMA':<10} {'Code':<15} {'Chip-level':<18} {'Soft (graceful)':<20} {'3G CDMA2000'}")
print(f"{'OFDMA':<10} {'Freq + Time':<15} {'Symbol-level':<18} {'Flexible':<20} {'4G LTE, 5G NR'}")

## Summary

### Key Concepts

| Concept | Formula / Definition |
|---------|---------------------|
| Processing gain | $G_p = R_c / R_b$ |
| m-sequence length | $L = 2^n - 1$ |
| Autocorrelation | $R(0) = 1$, $R(\tau \neq 0) = -1/L$ |
| Walsh orthogonality | $W_i \cdot W_j = 0$ for $i \neq j$ |
| CDMA detection | $\hat{d}_k = \text{sign}(r \cdot W_k / N)$ |

### Key Takeaways

1. **DSSS** trades bandwidth for SNR improvement (processing gain)
2. **m-sequences** have near-ideal autocorrelation for synchronization
3. **Walsh codes** give perfect orthogonality when synchronized
4. **CDMA** allows all users to share the same time and frequency
5. **Near-far problem** requires power control in practical systems
6. GPS uses CDMA: each satellite has a unique Gold code ($G_p \approx 43$ dB)

### Applications

| System | Type | Processing Gain |
|--------|------|----------------|
| GPS C/A | DSSS | 43 dB |
| IS-95 CDMA | DSSS | 21 dB |
| WCDMA (3G) | DSSS | Variable |
| 802.11b WiFi | DSSS/CCK | 11 dB |

### Next Topics
- **Lesson 20:** M-ary PSK, QAM, EVM, and FT8
- Constellation diagrams and spectral efficiency